In [7]:
import pandas as pd
import psycopg2
from dotenv import load_dotenv
import os



In [ ]:
load_dotenv("/home/nikita/projects/mle-project-sprint-1-v001/part2_dvc/.env")

DB_DESTINATION_HOST = os.getenv("DB_DESTINATION_HOST")
DB_DESTINATION_PORT = os.getenv("DB_DESTINATION_PORT")
DB_DESTINATION_NAME = os.getenv("DB_DESTINATION_NAME")
DB_DESTINATION_USER = os.getenv("DB_DESTINATION_USER")
DB_DESTINATION_PASSWORD = os.getenv("DB_DESTINATION_PASSWORD")


In [9]:


# задай параметры подключения
conn = psycopg2.connect(
    host=DB_DESTINATION_HOST,
    port=DB_DESTINATION_PORT,
    dbname=DB_DESTINATION_NAME,
    user=DB_DESTINATION_USER,
    password=DB_DESTINATION_PASSWORD
)

sql = """
SELECT
    f.id AS flat_id,
    f.building_id,
    f.floor,
    f.kitchen_area,
    f.living_area,
    f.rooms,
    f.is_apartment,
    f.studio,
    f.total_area,
    f.price,
    b.build_year,
    b.building_type_int,
    b.latitude,
    b.longitude,
    b.ceiling_height,
    b.flats_count,
    b.floors_total,
    b.has_elevator
FROM flats AS f
JOIN buildings AS b
    ON b.id = f.building_id
"""

df = pd.read_sql(sql, conn)

conn.close()

df.head()

/tmp/ipykernel_2816833/3401344159.py:35: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


,flat_id,building_id,floor,kitchen_area,living_area,rooms,is_apartment,studio,total_area,price,build_year,building_type_int,latitude,longitude,ceiling_height,flats_count,floors_total,has_elevator
0,0,6220,9,9.9,19.900000,1,False,False,35.099998,9500000,1965,6,55.717113,37.781120,2.64,84,12,True
1,1,18012,7,0.0,16.600000,1,False,False,43.000000,13500000,2001,2,55.794849,37.608013,3.00,97,10,True
2,2,17821,9,9.0,32.000000,2,False,False,56.000000,13500000,2000,4,55.740040,37.761742,2.70,80,10,True
3,3,18579,1,10.1,43.099998,3,False,False,76.000000,20000000,2002,4,55.672016,37.570877,2.64,771,17,True
4,4,9293,3,3.0,14.000000,1,False,False,24.000000,5200000,1971,1,55.808807,37.707306,2.60,208,9,True


In [10]:
df["is_apartment"].value_counts()

is_apartment
False    139990
True       1372
Name: count, dtype: int64

In [11]:
df["studio"].value_counts()

studio
False    141362
Name: count, dtype: int64

In [12]:
df["has_elevator"].value_counts()

has_elevator
True     126856
False     14506
Name: count, dtype: int64

In [13]:
is_duplicated_id = df.duplicated(subset=['flat_id'], keep=False)
print(sum(is_duplicated_id)) 



0


In [14]:
def remove_duplicates(data):
    feature_cols = data.columns.drop('flat_id').tolist()
    is_duplicated_features = data.duplicated(subset=feature_cols, keep=False)
    data = data[~is_duplicated_features].reset_index(drop=True)
    return data

In [15]:
df_no_duplicates = remove_duplicates(df)

In [16]:
print(len(df))
print(len(df_no_duplicates))

141362
123937


In [17]:
df_no_duplicates.isnull().sum() 

flat_id              0
building_id          0
floor                0
kitchen_area         0
living_area          0
rooms                0
is_apartment         0
studio               0
total_area           0
price                0
build_year           0
building_type_int    0
latitude             0
longitude            0
ceiling_height       0
flats_count          0
floors_total         0
has_elevator         0
dtype: int64

In [18]:
num_cols = df_no_duplicates.drop(columns=["is_apartment", "has_elevator"]).select_dtypes(['float']).columns
threshold = 1.5
potential_outliers = pd.DataFrame()

for col in num_cols:
    Q1 = df_no_duplicates[col].quantile(0.25)
    Q3 = df_no_duplicates[col].quantile(0.75)
    IQR = Q3 - Q1 
    margin = threshold*IQR
    lower = Q1 - margin
    upper =Q3 + margin 
    potential_outliers[col] = ~df_no_duplicates[col].between(lower, upper)

outliers = potential_outliers.any(axis=1)

df_no_duplicates[outliers]

,flat_id,building_id,floor,kitchen_area,living_area,rooms,is_apartment,studio,total_area,price,build_year,building_type_int,latitude,longitude,ceiling_height,flats_count,floors_total,has_elevator
31,34,15810,8,8.5,19.000000,1,False,False,38.0,5800000,1992,4,55.983387,37.152309,2.64,379,14,True
43,49,22115,22,12.0,60.000000,2,False,False,94.0,22450000,2012,2,55.827671,37.487568,3.10,327,40,True
52,58,24471,9,0.0,0.000000,3,False,False,126.0,39500000,2019,3,55.708664,37.590256,3.15,301,10,True
62,68,23784,5,0.0,0.000000,3,False,False,90.0,95000000,2017,2,55.760914,37.550209,3.20,105,12,True
74,80,23704,17,0.0,0.000000,6,False,False,282.0,206000000,2017,2,55.710697,37.607449,3.00,441,18,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
123916,141336,21544,23,0.0,160.000000,5,False,False,201.0,74500000,2010,2,55.715485,37.479034,3.00,183,24,True
123917,141337,21544,22,0.0,110.000000,5,False,False,179.0,82000000,2010,2,55.715485,37.479034,3.00,183,24,True
123918,141338,21544,24,0.0,110.000000,5,False,False,184.0,92000000,2010,2,55.715485,37.479034,3.00,183,24,True
123920,141340,22087,39,10.0,80.000000,2,False,False,112.0,45000000,2012,2,55.744308,37.419521,3.00,332,40,True


In [19]:
def remove_outliers_iqr(data: pd.DataFrame, threshold: float = 1.5) -> pd.DataFrame:
    num_cols = data.select_dtypes(include=["float", "int"]).columns
    mask = pd.Series(True, index=data.index)

    for col in num_cols:
        Q1 = data[col].quantile(0.25)
        Q3 = data[col].quantile(0.75)
        IQR = Q3 - Q1

        margin = threshold * IQR
        lower = Q1 - margin
        upper = Q3 + margin

        mask &= data[col].between(lower, upper)

    return data[mask]

In [20]:
df_no_duplicates_no_outliers = remove_outliers_iqr(df_no_duplicates)

In [21]:
df_no_duplicates_no_outliers

,flat_id,building_id,floor,kitchen_area,living_area,rooms,is_apartment,studio,total_area,price,build_year,building_type_int,latitude,longitude,ceiling_height,flats_count,floors_total,has_elevator
0,0,6220,9,9.90,19.900000,1,False,False,35.099998,9500000,1965,6,55.717113,37.781120,2.64,84,12,True
1,1,18012,7,0.00,16.600000,1,False,False,43.000000,13500000,2001,2,55.794849,37.608013,3.00,97,10,True
2,2,17821,9,9.00,32.000000,2,False,False,56.000000,13500000,2000,4,55.740040,37.761742,2.70,80,10,True
4,4,9293,3,3.00,14.000000,1,False,False,24.000000,5200000,1971,1,55.808807,37.707306,2.60,208,9,True
5,5,23964,9,0.00,0.000000,2,False,False,51.009998,8490104,2017,4,55.724728,37.743069,2.70,192,17,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
123931,141356,9503,8,6.00,42.000000,3,False,False,64.000000,10800000,1971,4,55.740402,37.834579,2.64,428,9,True
123933,141358,3162,5,5.28,28.330000,2,False,False,41.110001,7400000,1960,1,55.727470,37.768677,2.48,80,5,False
123934,141359,6513,7,5.30,20.000000,1,False,False,31.500000,9700000,1966,4,55.704315,37.506584,2.64,72,9,True
123935,141360,23952,15,13.80,33.700001,2,False,False,65.300003,11750000,2017,4,55.699863,37.939564,2.70,480,25,True


In [22]:
conn = psycopg2.connect(
    host=DB_DESTINATION_HOST,
    port=DB_DESTINATION_PORT,
    dbname=DB_DESTINATION_NAME,
    user=DB_DESTINATION_USER,
    password=DB_DESTINATION_PASSWORD
)



sql = """
SELECT *
FROM joined_data_housing
"""

df2 = pd.read_sql(sql, conn)

conn.close()

df2.head()

/tmp/ipykernel_2816833/2958719083.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df2 = pd.read_sql(sql, conn)


,flat_id,building_id,floor,kitchen_area,living_area,rooms,is_apartment,total_area,price,build_year,building_type_int,latitude,longitude,ceiling_height,flats_count,floors_total,has_elevator
0,0,6220,9,9.9,19.900000,1,0,35.099998,9500000,1965,6,55.717113,37.781120,2.64,84,12,1
1,1,18012,7,0.0,16.600000,1,0,43.000000,13500000,2001,2,55.794849,37.608013,3.00,97,10,1
2,2,17821,9,9.0,32.000000,2,0,56.000000,13500000,2000,4,55.740040,37.761742,2.70,80,10,1
3,3,18579,1,10.1,43.099998,3,0,76.000000,20000000,2002,4,55.672016,37.570877,2.64,771,17,1
4,4,9293,3,3.0,14.000000,1,0,24.000000,5200000,1971,1,55.808807,37.707306,2.60,208,9,1


In [23]:
df2

,flat_id,building_id,floor,kitchen_area,living_area,rooms,is_apartment,total_area,price,build_year,building_type_int,latitude,longitude,ceiling_height,flats_count,floors_total,has_elevator
0,0,6220,9,9.90,19.900000,1,0,35.099998,9500000,1965,6,55.717113,37.781120,2.64,84,12,1
1,1,18012,7,0.00,16.600000,1,0,43.000000,13500000,2001,2,55.794849,37.608013,3.00,97,10,1
2,2,17821,9,9.00,32.000000,2,0,56.000000,13500000,2000,4,55.740040,37.761742,2.70,80,10,1
3,3,18579,1,10.10,43.099998,3,0,76.000000,20000000,2002,4,55.672016,37.570877,2.64,771,17,1
4,4,9293,3,3.00,14.000000,1,0,24.000000,5200000,1971,1,55.808807,37.707306,2.60,208,9,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
141357,141357,22455,16,11.00,18.000000,1,0,42.000000,10500000,2013,4,55.626579,37.313503,2.64,672,25,1
141358,141358,3162,5,5.28,28.330000,2,0,41.110001,7400000,1960,1,55.727470,37.768677,2.48,80,5,0
141359,141359,6513,7,5.30,20.000000,1,0,31.500000,9700000,1966,4,55.704315,37.506584,2.64,72,9,1
141360,141360,23952,15,13.80,33.700001,2,0,65.300003,11750000,2017,4,55.699863,37.939564,2.70,480,25,1


In [24]:

conn = psycopg2.connect(
    host=DB_DESTINATION_HOST,
    port=DB_DESTINATION_PORT,
    dbname=DB_DESTINATION_NAME,
    user=DB_DESTINATION_USER,
    password=DB_DESTINATION_PASSWORD
)



sql = """
SELECT *
FROM cleaned_data_housing
"""

df_cleaned = pd.read_sql(sql, conn)

conn.close()

df_cleaned.head()

/tmp/ipykernel_2816833/2603271131.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_cleaned = pd.read_sql(sql, conn)


,id,flat_id,building_id,floor,kitchen_area,living_area,rooms,is_apartment,total_area,price,build_year,building_type_int,latitude,longitude,ceiling_height,flats_count,floors_total,has_elevator
0,1,0,6220,9,9.9,19.9,1,0,35.099998,9500000.0,1965,6,55.717113,37.781120,2.64,84,12,1
1,2,1,18012,7,0.0,16.6,1,0,43.000000,13500000.0,2001,2,55.794849,37.608013,3.00,97,10,1
2,3,2,17821,9,9.0,32.0,2,0,56.000000,13500000.0,2000,4,55.740040,37.761742,2.70,80,10,1
3,4,4,9293,3,3.0,14.0,1,0,24.000000,5200000.0,1971,1,55.808807,37.707306,2.60,208,9,1
4,5,5,23964,9,0.0,0.0,2,0,51.009998,8490104.0,2017,4,55.724728,37.743069,2.70,192,17,1


In [25]:
df_cleaned

,id,flat_id,building_id,floor,kitchen_area,living_area,rooms,is_apartment,total_area,price,build_year,building_type_int,latitude,longitude,ceiling_height,flats_count,floors_total,has_elevator
0,1,0,6220,9,9.90,19.900000,1,0,35.099998,9500000.0,1965,6,55.717113,37.781120,2.64,84,12,1
1,2,1,18012,7,0.00,16.600000,1,0,43.000000,13500000.0,2001,2,55.794849,37.608013,3.00,97,10,1
2,3,2,17821,9,9.00,32.000000,2,0,56.000000,13500000.0,2000,4,55.740040,37.761742,2.70,80,10,1
3,4,4,9293,3,3.00,14.000000,1,0,24.000000,5200000.0,1971,1,55.808807,37.707306,2.60,208,9,1
4,5,5,23964,9,0.00,0.000000,2,0,51.009998,8490104.0,2017,4,55.724728,37.743069,2.70,192,17,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
97173,97174,141356,9503,8,6.00,42.000000,3,0,64.000000,10800000.0,1971,4,55.740402,37.834579,2.64,428,9,1
97174,97175,141358,3162,5,5.28,28.330000,2,0,41.110001,7400000.0,1960,1,55.727470,37.768677,2.48,80,5,0
97175,97176,141359,6513,7,5.30,20.000000,1,0,31.500000,9700000.0,1966,4,55.704315,37.506584,2.64,72,9,1
97176,97177,141360,23952,15,13.80,33.700001,2,0,65.300003,11750000.0,2017,4,55.699863,37.939564,2.70,480,25,1
